In [1]:
# Metaconnectivity pipeline
# Notebook 00: compute + modularity + trimer indices + freeze artifact
# Finish point: results/mc_frozen/<dataset>/mc_frozen_....npz


In [2]:
from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np


In [3]:
from shared_code.fun_loaddata import load_timeseries_bundle
from shared_code.fun_paths import get_paths
from shared_code.fun_metaconnectivity import (
    compute_metaconnectivity,
    fun_allegiance_communities,
    intramodule_indices_mask,
    get_fc_mc_indices,
    get_mc_region_identities,
    # The following *might* exist in shared_code; if not, we'll adapt:
    build_trimer_mask,
    compute_trimers_identity,
)


[info] Using PATHS_ROOT=/media/samy/Elements2/Proyectos/LauraHarsan


In [4]:
# ---------------- Dataset ----------------
DATASET = "ines_abdallah"
TIMECOURSE_FOLDER = "Timecourses_updated_03052024"
COGNITIVE_FILE = "ROIs.xlsx"
ANAT_LABELS_FILE = "41_Allen.txt"

BUNDLE_NPZ = "ts_and_meta_2m4m.npz"
BUNDLE_GROUPING = "grouping_data_oip.pkl"

# ---------------- MC params ----------------
WINDOW_SIZE = 7
LAG = 1
N_JOBS = -1

# ---------------- Allegiance / modularity ----------------
N_RUNS_ALLEGIANCE = 1000
GAMMA_PT = 100

# ---------------- Reference group selection ----------------
REF_COL = 2
REF_ROW = 0

# ---------------- Tag for filenames ----------------
RUN_TAG = f"w={WINDOW_SIZE}_lag={LAG}_runs={N_RUNS_ALLEGIANCE}_gamma={GAMMA_PT}_ref={REF_COL}-{REF_ROW}"
print("RUN_TAG:", RUN_TAG)


RUN_TAG: w=7_lag=1_runs=1000_gamma=100_ref=2-0


In [5]:
paths = get_paths(
    dataset_name=DATASET,
    timecourse_folder=TIMECOURSE_FOLDER,
    cognitive_data_file=COGNITIVE_FILE,
    anat_labels_file=ANAT_LABELS_FILE,
)

bundle = load_timeseries_bundle(
    paths["preprocessed"] / BUNDLE_NPZ,
    paths["preprocessed"] / BUNDLE_GROUPING,
)

ts = bundle.ts
n_animals = bundle.n_animals
regions = bundle.n_regions
mask_groups = bundle.mask_groups
label_variables = bundle.label_variables

if mask_groups is None or label_variables is None:
    raise ValueError("Grouping data missing: expected mask_groups and label_variables in bundle.")

print("ts shape:", np.shape(ts))
print("n_animals:", n_animals, "regions:", regions)
print("n grouping columns:", len(mask_groups))


ts shape: (126, 450, 41)
n_animals: 126 regions: 41
n grouping columns: 4


In [6]:
label_ref = label_variables[REF_COL][REF_ROW]
ind_ref = mask_groups[REF_COL][REF_ROW]

print("Reference label:", label_ref)
print("Reference mask type:", type(ind_ref))


Reference label: wt 2m
Reference mask type: <class 'numpy.ndarray'>


In [ ]:
t0 = time.time()
mc = compute_metaconnectivity(
    ts,
    window_size=WINDOW_SIZE,
    lag=LAG,
    n_jobs=N_JOBS,
    save_path=None
)
t1 = time.time()

mc = np.asarray(mc)
print("MC shape:", mc.shape, f"(computed in {t1 - t0:.2f}s)")


[info] Using PATHS_ROOT=/media/samy/Elements2/Proyectos/LauraHarsan
[info] Using PATHS_ROOT=/media/samy/Elements2/Proyectos/LauraHarsan
[info] Using PATHS_ROOT=/media/samy/Elements2/Proyectos/LauraHarsan
[info] Using PATHS_ROOT=/media/samy/Elements2/Proyectos/LauraHarsan
[info] Using PATHS_ROOT=/media/samy/Elements2/Proyectos/LauraHarsan
[info] Using PATHS_ROOT=/media/samy/Elements2/Proyectos/LauraHarsan
[info] Using PATHS_ROOT=/media/samy/Elements2/Proyectos/LauraHarsan
[info] Using PATHS_ROOT=/media/samy/Elements2/Proyectos/LauraHarsan
here
Saving mc stream to: reports/metaconnectivity/ines_abdallah/mc/mc_window_size=7_lag=1_animals=126_regions=41.npz
MC shape: (126, 820, 820) (computed in 46.91s)


In [8]:
E_expected = regions * (regions - 1) // 2

if mc.ndim != 3:
    raise ValueError(f"Expected mc.ndim==3 (n_animals, E, E). Got shape={mc.shape}")

if mc.shape[0] != n_animals:
    raise ValueError(f"Expected mc.shape[0]==n_animals. Got {mc.shape[0]} vs {n_animals}")

if mc.shape[1] != mc.shape[2]:
    raise ValueError(f"Expected square E×E MC. Got {mc.shape}")

if mc.shape[1] != E_expected:
    raise ValueError(f"Expected E={E_expected} from regions={regions}. Got E={mc.shape[1]}")

if not np.isfinite(mc).all():
    bad = np.isnan(mc).sum() + np.isinf(mc).sum()
    raise ValueError(f"MC contains non-finite values: count={bad}")

print("MC validated: per-animal edge×edge metaconnectivity")


MC validated: per-animal edge×edge metaconnectivity


In [ ]:
mc_ref = np.mean(mc[ind_ref], axis=0)  # mean across animals in reference group

(
    mc_ref_allegiance_communities,
    allegiance_sort,
    contingency_matrix,
) = fun_allegiance_communities(
    mc_ref,
    n_runs=N_RUNS_ALLEGIANCE,
    gamma_pt=GAMMA_PT,
    save_path=Path("reports/metaconnectivity") / DATASET / "allegiance",
    ref_name=str(label_ref),
    n_jobs=N_JOBS,
)

print("allegiance_sort length:", len(allegiance_sort))
print("communities shape:", np.shape(mc_ref_allegiance_communities))


Running Louvain jobs:   6%|▌         | 5896/100000 [11:30<3:46:02,  6.94it/s]

In [ ]:
# Reorder MC by allegiance communities
mc_allegiance = mc[:, allegiance_sort][:, :, allegiance_sort]

# Fill diagonal with nan (as in your scripts)
idx_diag = np.arange(E_expected)
mc_allegiance[..., idx_diag, idx_diag] = np.nan

# Build module mask from communities
intramodules_idx, intramodule_indices, mc_modules_mask = intramodule_indices_mask(
    mc_ref_allegiance_communities
)

# Reorder mask to match allegiance sort
mc_modules_mask = mc_modules_mask[allegiance_sort][:, allegiance_sort]

print("mc_allegiance shape:", mc_allegiance.shape)
print("mc_modules_mask shape:", mc_modules_mask.shape)


mc_allegiance shape: (126, 820, 820)
mc_modules_mask shape: (820, 820)


In [ ]:
# Build basic indices (must be consistent with your helper)
fc_idx, mc_idx = get_fc_mc_indices(regions, allegiance_sort=allegiance_sort)

# Region identities per MC entry (which 4 regions define MC_[ij,kl])
mc_reg_idx, fc_reg_idx = get_mc_region_identities(fc_idx, mc_idx)

# Extract lower-triangle MC values in sorted space
mc_val = mc_allegiance[:, mc_idx[:, 0], mc_idx[:, 1]]  # (n_animals, E2)

# Module label per MC entry (0 inter-module, >0 intra-module module id)
mc_mod_idx = mc_modules_mask[mc_idx[:, 0], mc_idx[:, 1]].astype(int)

print("mc_val shape:", mc_val.shape)
print("mc_mod_idx shape:", mc_mod_idx.shape)
print("mc_idx shape:", mc_idx.shape, "fc_idx shape:", fc_idx.shape)
print("mc_reg_idx shape:", np.shape(mc_reg_idx), "fc_reg_idx shape:", np.shape(fc_reg_idx))


mc_val shape: (126, 335790)
mc_mod_idx shape: (335790,)
mc_idx shape: (335790, 2) fc_idx shape: (820, 2)
mc_reg_idx shape: (4, 335790) fc_reg_idx shape: (335790, 2, 2)


In [ ]:
# Compute trimer identities (region-level definition)
trimer_index, trimer_reg_id, trimer_apex = compute_trimers_identity(regions)

# Build trimer mask in edge-space (E×E), then reorder
mc_nplets_mask = build_trimer_mask(trimer_index, trimer_apex, E_expected)
mc_nplets_mask = mc_nplets_mask[allegiance_sort][:, allegiance_sort]

# Reduce to 1D per MC entry (same indexing as mc_val)
mc_nplets_index = mc_nplets_mask[mc_idx[:, 0], mc_idx[:, 1]]

print("mc_nplets_index shape:", mc_nplets_index.shape)
print("n trimers (mc_nplets_index>0):", int(np.sum(mc_nplets_index > 0)))
print("n tetramers (==0):", int(np.sum(mc_nplets_index == 0)))


mc_nplets_index shape: (335790,)
n trimers (mc_nplets_index>0): 31980
n tetramers (==0): 303810


In [ ]:
# ---------------------------------------------
# Output directories (repo conventions)
# ---------------------------------------------
mc_mod_dir = paths["mc_mod"]
trimers_dir = paths.get("trimers", paths["results"] / "trimers")

mc_mod_dir.mkdir(parents=True, exist_ok=True)
trimers_dir.mkdir(parents=True, exist_ok=True)

print("[OK] Output dirs:")
print("  mc_mod   →", mc_mod_dir)
print("  trimers  →", trimers_dir)

params = dict(
    dataset=paths["results"].name,
    window_size=WINDOW_SIZE,
    lag=LAG,
    n_runs_allegiance=N_RUNS_ALLEGIANCE,
    gamma_pt=GAMMA_PT,
    ref_col=REF_COL,
    ref_row=REF_ROW,
    n_animals=int(n_animals),
    n_regions=int(regions),
)

np.savez_compressed(
    mc_mod_dir / f"mc_frozen_{RUN_TAG}.npz",
    mc_val_tril=mc_val,
    mc_idx_tril=mc_idx,
    mc_mod_idx=mc_mod_idx,
    mc_reg_idx=mc_reg_idx,
    mc_nplets_index=mc_nplets_index,
    params=params,
)

print("Saving to:")
print("  mc_mod:", mc_mod_dir)
print("  trimers:", trimers_dir)


[OK] Output dirs:
  mc_mod   → /media/samy/Elements2/Proyectos/LauraHarsan/results/ines_abdallah/mc_mod
  trimers  → /media/samy/Elements2/Proyectos/LauraHarsan/results/ines_abdallah/trimers
Saving to:
  mc_mod: /media/samy/Elements2/Proyectos/LauraHarsan/results/ines_abdallah/mc_mod
  trimers: /media/samy/Elements2/Proyectos/LauraHarsan/results/ines_abdallah/trimers
